# Notebook B4: Minor-embedding và độ dài chain

Đi kèm [B-05: Minor-embedding và chain](../../docs/B-extensions/05-embedding-and-chains.md). Notebook dùng `minorminer` để nhúng các đồ thị lên topology cỡ thật của máy D-Wave (Chimera 2048 qubit, Pegasus 5640 qubit, Zephyr 4800 qubit). **Không cần tài khoản D-Wave**, mọi thứ chạy cục bộ.

In [1]:
import time, warnings
warnings.filterwarnings("ignore")
import networkx as nx
import minorminer
import dwave_networkx as dnx

topologies = {
    "Chimera C16": dnx.chimera_graph(16),
    "Pegasus P16": dnx.pegasus_graph(16),
    "Zephyr Z12":  dnx.zephyr_graph(12),
}
for name, G in topologies.items():
    degs = [d for _, d in G.degree()]
    print(f"{name}: {G.number_of_nodes()} qubit, {G.number_of_edges()} coupler, bậc tối đa {max(degs)}")

Chimera C16: 2048 qubit, 6016 coupler, bậc tối đa 6
Pegasus P16: 5640 qubit, 40484 coupler, bậc tối đa 15
Zephyr Z12: 4800 qubit, 45864 coupler, bậc tối đa 20


## 1. Đồ thị đầy đủ $K_n$ (QUBO dày)

Một QUBO mà mọi cặp biến đều có tương tác bậc hai (ví dụ bài toán có ràng buộc one-hot dày) có đồ thị $K_n$. Ta nhúng $K_n$ và đếm tổng số qubit vật lý cùng độ dài chain lớn nhất.

In [2]:
def embed_stats(graph_edges, target, timeout=30, seed=1):
    t = time.time()
    emb = minorminer.find_embedding(graph_edges, target.edges, timeout=timeout, random_seed=seed)
    if not emb:
        return None
    lens = [len(c) for c in emb.values()]
    return sum(lens), max(lens), time.time() - t

print("Nhúng K_n; mỗi ô: (tổng qubit vật lý, chain dài nhất)")
sizes = (8, 16, 24, 32)
print(f"{'':12s}" + "".join(f"K{n:<14d}" for n in sizes))
for name, G in topologies.items():
    cells = []
    for n in sizes:
        r = embed_stats(nx.complete_graph(n).edges, G)
        cells.append("không tìm được" if r is None else f"({r[0]}, {r[1]})")
    print(f"{name:12s}" + "".join(f"{c:<15s}" for c in cells))

Nhúng K_n; mỗi ô: (tổng qubit vật lý, chain dài nhất)
            K8             K16            K24            K32            


Chimera C16 (22, 3)        (80, 6)        (188, 10)      (339, 13)      


Pegasus P16 (12, 2)        (38, 3)        (80, 4)        (140, 6)       


Zephyr Z12  (12, 2)        (37, 3)        (63, 3)        (110, 5)       


Số qubit vật lý tăng nhanh hơn tuyến tính theo $n$, và chain dài hơn nhiều trên Chimera so với Pegasus/Zephyr vì mỗi qubit chỉ nối được tối đa 6 qubit khác (bậc 6, so với 15 và 20). Đây là lý do độ kết nối của topology quan trọng, và là lý do các bài toán dày bị giới hạn ở kích thước nhỏ hơn rất nhiều so với tổng số qubit.

## 2. Đồ thị thưa: 3-regular ngẫu nhiên

Nhiều bài toán tối ưu tổ hợp thực tế (ví dụ Max-Cut trên đồ thị thưa) có đồ thị bậc nhỏ, và nhúng dễ hơn nhiều.

In [3]:
print("Nhúng đồ thị 3-regular ngẫu nhiên; mỗi ô: (tổng qubit vật lý, chain dài nhất)")
sizes = (32, 64, 128, 256)
print(f"{'':12s}" + "".join(f"n={n:<13d}" for n in sizes))
for name, G in topologies.items():
    cells = []
    for n in sizes:
        H = nx.random_regular_graph(3, n, seed=1)
        r = embed_stats(H.edges, G)
        cells.append("không tìm được" if r is None else f"({r[0]}, {r[1]})")
    print(f"{name:12s}" + "".join(f"{c:<15s}" for c in cells))

Nhúng đồ thị 3-regular ngẫu nhiên; mỗi ô: (tổng qubit vật lý, chain dài nhất)
            n=32           n=64           n=128          n=256          


Chimera C16 (63, 3)        (147, 4)       (389, 8)       (1205, 15)     


Pegasus P16 (33, 2)        (72, 2)        (175, 3)       (497, 5)       


Zephyr Z12  (32, 1)        (70, 2)        (149, 2)       (469, 5)       


Đồ thị thưa với hàng trăm biến vẫn nhúng được: trên Pegasus và Zephyr chain dài tối đa 5 ở $n = 256$ (Chimera cần chain tới 15), trong khi $K_{32}$ đã cần 339 qubit trên Chimera. Kết luận thực tế: **cấu trúc đồ thị của QUBO quyết định bài toán có vừa với máy hay không**, quan trọng hơn cả số biến.

Lưu ý: `find_embedding` là heuristic ngẫu nhiên, nên số liệu có thể thay đổi chút ít theo hạt giống; đây là các cận trên của một lần thử, không phải nhúng tối ưu (bài toán minor-embedding là NP-hard như bài báo nêu).

## 3. Đồ thị của bài toán TSP (Ví dụ 8)

Với $n$ thành phố, ta có $n^2$ biến nhị phân $x_{a,j}$ (thành phố $a$ ở vị trí $j$). Hai ràng buộc hoán vị (23), (24) tạo các tương tác bậc hai giữa mọi cặp biến cùng hàng (cùng thành phố) và cùng cột (cùng vị trí). Phần chi phí quãng đường $\sum d_{a,b}\,x_{a,j}\,x_{b,j+1}$ nối thêm mọi cặp ở hai vị trí kề nhau. Kết quả là mỗi biến có bậc khoảng $4(n-1)$.

In [4]:
def tsp_graph(n):
    G = nx.Graph()
    G.add_nodes_from((a, j) for a in range(n) for j in range(n))
    for j in range(n):
        for a in range(n):
            for b in range(a + 1, n):
                G.add_edge((a, j), (b, j))                  # mỗi vị trí đúng một thành phố
    for a in range(n):
        for j in range(n):
            for k in range(j + 1, n):
                G.add_edge((a, j), (a, k))                  # mỗi thành phố đúng một vị trí
    for j in range(n):
        for a in range(n):
            for b in range(n):
                if a != b:
                    G.add_edge((a, j), (b, (j + 1) % n))    # chi phí quãng đường giữa hai vị trí kề nhau
    return G

print("n   biến logic  bậc tối đa  | " + " | ".join(f"{name:22s}" for name in topologies))
for n_cities in (4, 6, 8, 10, 12):
    H = tsp_graph(n_cities)
    max_deg = max(d for _, d in H.degree())
    cells = []
    for name, G in topologies.items():
        r = embed_stats(H.edges, G, timeout=30)
        cells.append("không tìm được" if r is None else f"{r[0]:5d} qubit, chain<={r[1]:2d}")
    print(f"{n_cities:2d}  {H.number_of_nodes():8d}  {max_deg:9d}   | " + " | ".join(f"{c:22s}" for c in cells))

n   biến logic  bậc tối đa  | Chimera C16            | Pegasus P16            | Zephyr Z12            


 4        16         12   |    75 qubit, chain<= 6 |    34 qubit, chain<= 3 |    32 qubit, chain<= 3


 6        36         20   |   359 qubit, chain<=14 |   154 qubit, chain<= 6 |   123 qubit, chain<= 5


 8        64         28   |  1084 qubit, chain<=26 |   445 qubit, chain<=10 |   367 qubit, chain<= 8
10       100         36   | không tìm được         |  1183 qubit, chain<=20 |   900 qubit, chain<=14


12       144         44   | không tìm được         |  2430 qubit, chain<=28 |  2009 qubit, chain<=25


Mỗi biến logic của TSP có bậc cỡ $4(n-1)$, vượt xa bậc tối đa của các qubit (6, 15, 20). Vì vậy bài toán đã cần chain dài và tổng số qubit vật lý lớn hơn rất nhiều số biến logic $n^2$, ngay từ các thể hiện rất nhỏ. Ở đây $n = 12$ thành phố (144 biến logic) đã chiếm khoảng 2000 qubit trên Pegasus P16, và heuristic không tìm được nhúng nào cho $n \ge 10$ trên Chimera C16 trong thời hạn đặt ra (kết quả "không tìm được" nghĩa là heuristic thất bại trong thời hạn, chưa phải chứng minh không tồn tại nhúng). Tốc độ tăng này gợi ý rằng TSP cỡ vài chục thành phố sẽ vượt quá khả năng của một chip đơn lẻ nếu mã hóa trực tiếp theo cách này; đó là ngoại suy thô từ các điểm $n \le 12$, không phải phép đo.